# Profile the candidate list and what TMDb returned

**Investigation, not production code.** This notebook profiles the *raw* inputs independently of the
pipeline: it reads `data/id_list.csv` as text and the cached TMDb responses in `data/raw/`, and it
**never reads `outputs/`** except in the last section, which exists only to compare the two routes.
When this notebook and the pipeline agree on a number, that is two independent routes to the same answer.

Every finding in `data_diagnosis.md` points to the section here that shows its number.

**To use it:** run `python run.py` once (so the cache exists), then *Run All*.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "explore" else Path.cwd()
RAW = ROOT / "data" / "raw"
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
print("Project folder:", ROOT)

## 1. The candidate list, exactly as received

Every value is read as text with no missing-value guessing (`keep_default_na=False`) and blank lines
kept, so nothing is converted or hidden. Line numbers match the file, with line 1 as the header.

In [ ]:
id_path = ROOT / "data" / "id_list.csv"
raw = pd.read_csv(id_path, dtype=str, keep_default_na=False, skip_blank_lines=False)
raw.insert(0, "file_line", np.arange(2, len(raw) + 2))
v = raw["tmdb_id"].str.strip()
raw_bytes = id_path.read_bytes()
print(f"{len(raw)} data lines, {raw.shape[1] - 1} column(s): {list(raw.columns[1:])}")
print(f"Line endings: {raw_bytes.count(b'\r\n')} Windows (CRLF) of {raw_bytes.count(b'\n')} total")
raw.head()

## 2. The list's claims, checked

The brief says "one column of TMDb movie ids". One line per question, each answered with a number.

In [ ]:
digits = v.str.fullmatch(r"\d+")
number = pd.to_numeric(v.where(v.str.fullmatch(r"-?\d+")), errors="coerce")
ids = number.where(number.gt(0))
pd.Series({
    "data lines": len(v),
    "blank lines": int(v.eq("").sum()),
    "values with surrounding spaces": int(raw["tmdb_id"].ne(v).sum()),
    "positive whole numbers": int((digits & number.gt(0)).sum()),
    "zero or negative numbers": int(number.le(0).sum()),
    "IMDb-format ids (tt + digits)": int(v.str.fullmatch(r"tt\d{7,8}").sum()),
    "other text": int((~v.eq("") & ~v.str.fullmatch(r"-?\d+") & ~v.str.fullmatch(r"tt\d{7,8}")).sum()),
    "distinct positive ids": int(ids.nunique()),
    "lines repeating an earlier id": int(ids.dropna().duplicated().sum()),
    "ids above 1,000,000": int(ids.gt(1_000_000).sum()),
    "largest id": int(ids.max()),
}, name="count").to_frame()

### The rows behind the numbers

In [ ]:
# Lines that are not a positive whole number
raw[~(digits & number.gt(0))]

In [ ]:
# Repeated ids, side by side with the line they repeat
raw.assign(id=ids)[ids.duplicated(keep=False) & ids.notna()].sort_values(["id", "file_line"])

In [ ]:
# Unusually large ids: real TMDb movie ids on this list are otherwise far smaller
raw.assign(id=ids)[ids.gt(1_000_000)]

## 3. What TMDb answered

From the cached responses (one file per endpoint, all ids). Each response carries the time it was fetched
and the HTTP status TMDb returned.

In [ ]:
def load_bundle(endpoint):
    b = json.loads((RAW / f"bundle_{endpoint}.json").read_text(encoding="utf-8"))
    return {k: p for k, p in b.items() if not k.startswith("_")}

movie_b, credits_b = load_bundle("movie"), load_bundle("credits")
status = pd.DataFrame({
    "movie_status": {int(k): (p or {}).get("_http_status") for k, p in movie_b.items()},
    "credits_status": {int(k): (p or {}).get("_http_status") for k, p in credits_b.items()},
    "fetched_at": {int(k): (p or {}).get("_fetched_at") for k, p in movie_b.items()},
})
print(f"ids asked for: {len(status)}")
print(status[["movie_status", "credits_status"]].value_counts().to_string())
print("fetched between", status["fetched_at"].min(), "and", status["fetched_at"].max())

In [ ]:
# Which ids TMDb did not recognise (compare with the large ids in section 2)
status[status["movie_status"].ne(200)]

In [ ]:
# The IMDb-format id: what TMDb's lookup translated it to
for f in sorted((RAW / "find").glob("*.json")):
    hits = (json.loads(f.read_text(encoding="utf-8"))["body"] or {}).get("movie_results", [])
    print(f.stem, "->", [(h["id"], h.get("title")) for h in hits])

## 4. Every film field: blanks, zeros, distinct values, range

In [ ]:
movies = pd.json_normalize([p["body"] for p in movie_b.values() if p and p["body"]], max_level=1)
cols = ["id", "title", "release_date", "status", "budget", "revenue", "runtime", "vote_average",
        "vote_count", "original_language", "adult", "video", "belongs_to_collection.id"]
cols = [c for c in cols if c in movies.columns]
num = movies[cols].apply(pd.to_numeric, errors="coerce")
pd.DataFrame({
    "missing": movies[cols].isna().sum() + movies[cols].astype(str).eq("").sum(),
    "zero": num.eq(0).sum(),
    "distinct": movies[cols].nunique(),
    "min": num.min(),
    "max": num.max(),
})

## 5. The claims we must check, each answered with a number

The questions an uncleaned, three-source list makes us ask. Zero is an answer too: it means
"checked, not present", which is evidence.

In [ ]:
m = movies.copy()
m["year"] = pd.to_datetime(m["release_date"].replace("", pd.NA), errors="coerce").dt.year
coll = m["belongs_to_collection.id"] if "belongs_to_collection.id" in m else pd.Series(np.nan, index=m.index)
norm = m["title"].str.casefold().str.replace(r"[^0-9a-z]+", " ", regex=True).str.strip()
ratio = m["revenue"] / m["budget"].where(m["budget"].gt(0))
requested = pd.Series([int(k) for k, p in movie_b.items() if p and p["body"]])
pd.Series({
    "films returned": len(m),
    "returned id differs from the id asked for": int((m["id"].to_numpy() != requested.to_numpy()).sum()),
    "budget = 0": int(m["budget"].eq(0).sum()),
    "revenue = 0": int(m["revenue"].eq(0).sum()),
    "budget between $1 and $999": int(m["budget"].between(1, 999).sum()),
    "revenue between $1 and $999": int(m["revenue"].between(1, 999).sum()),
    "no release date": int(m["year"].isna().sum()),
    "status other than Released": int(m["status"].ne("Released").sum()),
    "runtime under 40 minutes": int(m["runtime"].lt(40).sum()),
    "adult or video release": int((m["adult"].astype(bool) | m["video"].astype(bool)).sum()),
    "fewer than 50 votes": int(m["vote_count"].lt(50).sum()),
    "same title and year, different id": int((norm + "|" + m["year"].astype(str)).duplicated(keep=False).sum()),
    "same IMDb id, different TMDb id": int(m["imdb_id"].replace("", pd.NA).dropna().duplicated(keep=False).sum()),
    "same title, different year": int((norm.duplicated(keep=False) & ~(norm + "|" + m["year"].astype(str)).duplicated(keep=False)).sum()),
    "revenue more than 50x budget": int(ratio.gt(50).sum()),
    "released before 2000": int(m["year"].lt(2000).sum()),
    "in a collection (franchise)": int(coll.notna().sum()),
    "no collection (standalone)": int(coll.isna().sum()),
    "distinct collections": int(coll.nunique()),
}, name="count").to_frame()

### The rows behind the numbers

In [ ]:
# Extreme returns: real blockbusters, or unit errors?
m.assign(ratio=ratio.round(1))[ratio.gt(50)][["id", "title", "year", "budget", "revenue", "ratio"]]

In [ ]:
# Same title in different years: remakes, or a normaliser artefact?
m[norm.duplicated(keep=False)][["id", "title", "year", "belongs_to_collection.id"]].sort_values("title")

In [ ]:
# Standalone films (no collection): check none is obviously part of a series
m[coll.isna()][["id", "title", "year", "budget", "revenue", "vote_average"]].sort_values("year")

In [ ]:
# How many of each collection's films are on the list: a bundle with 1 film is not much of a bundle
(m.assign(coll_name=m.get("belongs_to_collection.name"))
  .dropna(subset=["coll_name"]).groupby("coll_name").size().sort_values().rename("films_on_list")
  .to_frame().T)

In [ ]:
# Year spread: how much of the list sits in the window Finance distrusts?
m["year"].lt(2000).map({True: "before 2000", False: "2000 on"}).groupby(coll.isna().map(
    {True: "standalone", False: "franchise"})).value_counts().unstack()

## 6. Credits: cast and crew

Read from the complete per-film files, not the trimmed copy the analysis uses.

In [ ]:
# Complete credits from the per-film cache files. The bundle the analysis reads keeps only
# directors and the top 10 cast, so it would not show what TMDb actually sent.
cr = [b for b in (json.loads(f.read_text(encoding="utf-8"))["body"]
                  for f in sorted((RAW / "credits").glob("*.json"))) if b]
cast = pd.json_normalize(cr, record_path="cast", meta=["id"], meta_prefix="film_")
crew = pd.json_normalize(cr, record_path="crew", meta=["id"], meta_prefix="film_")
per_film = pd.DataFrame({
    "cast": cast.groupby("film_id").size(),
    "crew": crew.groupby("film_id").size(),
    "directors": crew[crew["job"].eq("Director")].groupby("film_id")["id"].nunique(),
}).reindex([c["id"] for c in cr]).fillna(0)
per_film.describe().loc[["count", "min", "50%", "max"]]

In [ ]:
pd.Series({
    "films with no cast listed": int(per_film["cast"].eq(0).sum()),
    "films with no director": int(per_film["directors"].eq(0).sum()),
    "films with 2+ directors": int(per_film["directors"].ge(2).sum()),
    "revenue inflation if summed on the cast table (x)": round(
        cast.merge(m[["id", "revenue"]], left_on="film_id", right_on="id")["revenue"].sum()
        / m["revenue"].sum(), 1),
}, name="value").to_frame()

## 7. Cross-check against the pipeline

The only cell that reads `outputs/`. If the two routes disagree, one of them is wrong.

In [ ]:
led = pd.read_csv(ROOT / "outputs" / "ledger.csv")
films = pd.read_csv(ROOT / "outputs" / "films.csv")
checks = pd.DataFrame({
    "notebook": [len(v), int(ids.nunique()), int((status["movie_status"] == 200).sum()),
                 int(coll.notna().sum()), int(coll.isna().sum())],
    "pipeline": [len(led), int(led["id_status"].eq("candidate").sum()),
                 len(films),
                 int(films["collection_id"].notna().sum()), int(films["collection_id"].isna().sum())],
}, index=["lines in the list", "distinct positive ids", "films found",
          "franchise films", "standalone films"])
checks["agree"] = checks["notebook"].eq(checks["pipeline"])
checks

Note: "distinct positive ids" counts numbers written in the file. The pipeline's candidate count
can be higher by the IMDb-format id it resolves (section 3), when that resolves to a new film.